# BPL Sets and Searching

In [1]:
!pip install -U basedpl

In [2]:
%load_ext basedpl.notebooks

In [3]:
import basedpl
basedpl.__version__

'0.1.91'

## Is it there, and where?

Two everyday questions about lists: **is this item in there?** and **where is it?** This dialog answers both, then uses them for a shopping bill. After that come *set* operations, which combine two lists: everything in either, what they share, and what one has that the other doesn't. It ends with sorting scores into grade bands.

BPL docs: [Glyphs](https://answerdotai.github.io/basedpl/glyphs.html)

Glyphs and ideas this dialog teaches:

| Glyph | Plain English |
|---|---|
| `a∊b` | member: is each item of `a` in `b`? |
| `aιb` | index of: where is each item of `b` in `a`? A missing item gives the length of `a` |
| `a∪b` | union: everything in either, each once |
| `∪a` | unique: each different item once |
| `a∩b` | intersection: the items of `a` that are also in `b` |
| `a~b` | without: the items of `a` that are not in `b` |
| `≠a` | unique mask: `$t` where an item appears for the first time |
| `cuts⍸x` | interval index: how many of the cut-offs each item has reached |

Other glyphs used along the way:

| Glyph | Plain English |
|---|---|
| `←` | store a value in a name |
| `⍝` | comment: BPL ignores the rest of the line |
| `"…"` | text; `"a" "b"` is a list of words |
| `⊂` | wrap one word up as a single item |
| `#` | keep the items where the answer is `$t` |
| `~` (one argument) | not: swaps `$t` and `$f` |
| `v.[i]` | items of `v` at positions `i` |
| `,` | join |
| `+/` | total, or a count of the `$t`s |
| `≢` | count the items |
| `ₓ` | (in output) marks an exact number |

### The data

A shop's fruit and their prices (pence), and a shopping basket. Note that the basket has a `"kiwi"`, which the shop doesn't sell.

In [4]:
%%bpl
fruit←"apple" "pear" "fig" "plum"
price←30 45 20 60                   ⍝ one price per fruit
basket←"pear" "fig" "pear" "kiwi"
fruit
price
basket

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────────────────────┐
│ "apple" "pear" "fig" "plum" │
└∊────────────────────────────┘

30 45 20 60

┌→───────────────────────────┐
│ "pear" "fig" "pear" "kiwi" │
└∊───────────────────────────┘

### Is it there?

#### 1. Membership: `∊`

- `a∊b` asks, for **each item of `a`**, "is it in `b`?" The answer has one `$t` or `$f` per item of `a`.
- With letters: which letters of "banana" are vowels?
- With words: which basket items does the shop sell? Here each item is a whole word.

In [5]:
%%bpl
"banana"∊"aeiou"     ⍝ one answer per letter
basket∊fruit         ⍝ one answer per word: kiwi isn't sold

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

$f $t $f $t $f $t

$t $t $t $f

#### 2. One word on its own needs `⊂`

- `"fig"` on its own is a list of 3 letters, so `"fig"∊fruit` would ask about each **letter**.
- Wrap it with `⊂` to ask about the whole word.

In [6]:
%%bpl
(⊂"fig")∊fruit       ⍝ is the word "fig" sold? $t
["fig"]∊fruit       ⍝ is the word "fig" sold? $t

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

$t

[$t]

In [ ]:
%%prompt
brackets seems to work too?

Yes—both work here, but they create slightly different things:

- `⊂"fig"` wraps `"fig"` as **one scalar item**, so the result is scalar `$t`.
- `["fig"]` creates a **one-item list**, so the result is the one-item list `[$t]`.

That difference appears in your output:

```bpl
(⊂"fig")∊fruit   ⍝ $t
["fig"]∊fruit    ⍝ [$t]
```

Use `⊂"fig"` when you mean **the single word**. Use brackets when you deliberately want a list—especially for several words:

```bpl
["fig","pear"]∊fruit   ⍝ [$t,$t]
```

```json {.usage}
{"model": "gpt-5.6-sol", "prompt_tokens": 40223, "completion_tokens": 407, "total_tokens": 40630, "cached_tokens": 3968, "reasoning_tokens": 243, "cost": 0.0063387}
```


#### 3. Keep the items that are (or aren't) there

- Use the yes/no answers with `#` to keep items.
- `~` with one argument means "not": it swaps `$t` and `$f`. So `(~basket∊fruit)#basket` keeps what the shop **doesn't** sell.

In [7]:
%%bpl
(basket∊fruit)#basket       ⍝ the items the shop sells
(~basket∊fruit)#basket      ⍝ the items it doesn't

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→────────────────────┐
│ "pear" "fig" "pear" │
└∊────────────────────┘

┌→───────┐
│ "kiwi" │
└∊───────┘

### Where is it?

#### 4. Index of: `ι`

- `fruitι⊂"pear"` gives the position of `"pear"` in `fruit`: 1 (counting from 0).
- With a list on the right, you get one position for each item.
- A missing item gets the position just past the end: here 4, the length of `fruit`.

In [8]:
%%bpl
fruitι⊂"pear"           ⍝ 1
fruitιbasket            ⍝ 1 2 1 4: kiwi is missing, so 4

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1ₓ

[1 2 1 4]ₓ

**Walkthrough**

- `fruitι⊂"pear"`: "Where in `fruit` is the word `"pear"`?" `⊂` keeps `"pear"` as one word. Counting starts at 0 (apple 0, pear 1), so the answer is **1**. `ₓ` just means an exact number.
- `fruitιbasket`: "Where is each basket item in `fruit`?" pear → 1, fig → 2, pear → 1, kiwi → **4**.
- **Why 4?** `fruit` has positions 0–3. Not found gives the next number, which is the length of the list. So an answer equal to the length means "not found".
- **Order:** with `ι`, the list you search goes on the left. With `∊`, the things you're asking about go on the left.

#### 5. Look up the prices

- Store the positions, then read the prices at those positions: `price.[j]`.
- But the kiwi's position (4) is past the end of `price`. A neat fix is to add one extra price at the end, a 0 for "not sold". Then position 4 reads 0.
- `+/` gives the bill.

In [9]:
%%bpl
j←fruitιbasket       ⍝ where each basket item is
j
p←price,0            ⍝ prices, plus 0 for anything not sold
p.[j]                ⍝ the price of each item
+/p.[j]              ⍝ the bill: 110p

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

[1 2 1 4]ₓ

45 20 45 0

110

### Sets

#### 6. Everything, once: `∪`

- `∪x` gives each different item once, in order of first appearance.
- `a∪b` joins two lists, keeping each item once.

In [10]:
%%bpl
a←1 2 3 4 5
b←4 5 6 7
∪3 1 3 2 1       ⍝ 3 1 2
a∪b              ⍝ everything in a or b

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

3 1 2

1 2 3 4 5 6 7

#### 7. What they share: `∩`

- `a∩b` keeps the items of `a` that are also in `b`.
- With text, it keeps the left word's letters, repeats included, so the order of the two words matters.

In [11]:
%%bpl
a
b
a∩b                  ⍝ 4 5
"hello"∩"world"      ⍝ "llo": letters of hello found in world
"world"∩"hello"      ⍝ "ol": letters of world found in hello

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1 2 3 4 5

4 5 6 7

4 5

"llo"

"ol"

#### 8. What one has and the other doesn't: `~`

- With two arguments, `a~b` keeps the items of `a` that are **not** in `b`.
- So `basket~fruit` is a shorter way to write step 3's "not sold" line.

In [12]:
%%bpl
a
b
a~b                  ⍝ 1 2 3
b~a                  ⍝ 6 7
basket~fruit         ⍝ what the shop doesn't sell
"banana"~"aeiou"     ⍝ the consonants

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

1 2 3 4 5

4 5 6 7

1 2 3

6 7

┌→───────┐
│ "kiwi" │
└∊───────┘

"bnn"

#### 9. First appearances: `≠`

- With one argument, `≠x` marks each item `$t` the **first** time it appears, and `$f` after that.
- Adding up the marks counts the different items, the same as `≢∪x`.

In [13]:
%%bpl
x←3 1 3 2 1
≠x               ⍝ $t $t $f $t $f
+/≠x             ⍝ 3 different items
∪x
≢∪x              ⍝ the same

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

$t $t $f $t $f

3ₓ

3 1 2

3ₓ

### Sorting into bands

#### 10. Grade bands: `⍸`

- Below 50 is a fail, 50 to 69 a pass, and 70 or more a merit. The cut-offs are `50 70`.
- `cuts⍸scores` gives, for each score, **how many cut-offs it has reached**: 0, 1 or 2.
- That's exactly a position in the list of grade names.

In [14]:
%%bpl
cuts←50 70
scores←45 72 50 69 90
k←cuts⍸scores         ⍝ cut-offs reached: 0 2 1 1 2
k
grades←"fail" "pass" "merit"
grades
grades.[k]            ⍝ the grade for each score

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

[0 2 1 1 2]ₓ

┌→──────────────────────┐
│ "fail" "pass" "merit" │
└∊──────────────────────┘

┌→─────────────────────────────────────┐
│ "fail" "merit" "pass" "pass" "merit" │
└∊─────────────────────────────────────┘

### Your turn

1. Find the vowels in `"programming"`.
   *Hint:* `∊` and `#`.
2. Remove the spaces from `"the cat sat"`.
   *Hint:* `~` with `" "`.
3. Find the position of `"plum"` in `fruit`.
   *Hint:* don't forget `⊂`.
4. Work out the bill for a basket of `"apple" "apple" "plum"`. (It should be 120.)
   *Hint:* copy step 5.
5. Find the grades for the scores `55 65 75`.